# Food Demand Forecasting - Exploratory Data Analysis

This notebook performs comprehensive EDA on the food demand dataset to understand:
- Seasonal patterns and trends
- SKU-specific demand characteristics
- Autocorrelation and lag dependencies
- Anomaly detection and data quality

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Data Loading and Initial Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('../data/food_demand_data.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Number of weeks: {df['week'].nunique()}")
print(f"Number of SKUs: {df['sku'].nunique()}")
print(f"SKUs: {sorted(df['sku'].unique())}")

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values and data quality
print("Missing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)

## 2. Overall Demand Patterns

In [ ]:
# Total weekly demand across all SKUs
weekly_total = df.groupby(['week', 'date'])['demand'].sum().reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=weekly_total['date'], y=weekly_total['demand'],
                        mode='lines+markers', name='Total Weekly Demand'))
fig.update_layout(title='Total Weekly Demand Over Time',
                 xaxis_title='Date', yaxis_title='Total Demand')
fig.show()

In [ ]:
# Demand by SKU over time
fig = px.line(df, x='date', y='demand', color='sku',
              title='Weekly Demand by SKU Over Time')
fig.update_layout(height=600)
fig.show()

## 3. Seasonal Analysis

In [ ]:
# Seasonal patterns by SKU
seasonal_analysis = df.groupby(['sku', 'season'])['demand'].agg(['mean', 'std']).round(2)
print("Average demand by season and SKU:")
print(seasonal_analysis)

In [ ]:
# Seasonal heatmap
seasonal_pivot = df.groupby(['sku', 'season'])['demand'].mean().unstack()
plt.figure(figsize=(10, 8))
sns.heatmap(seasonal_pivot, annot=True, cmap='YlOrRd', fmt='.0f')
plt.title('Average Demand by SKU and Season')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly patterns
monthly_demand = df.groupby('month')['demand'].mean()
plt.figure(figsize=(12, 6))
monthly_demand.plot(kind='bar')
plt.title('Average Demand by Month (All SKUs)')
plt.xlabel('Month')
plt.ylabel('Average Demand')
plt.xticks(rotation=0)
plt.show()

## 4. Time Series Decomposition

In [ ]:
# Select top 4 SKUs by average demand for detailed analysis
top_skus = df.groupby('sku')['demand'].mean().nlargest(4).index.tolist()
print(f"Top 4 SKUs by average demand: {top_skus}")

# Perform seasonal decomposition for each top SKU
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
fig.suptitle('Seasonal Decomposition - Top 4 SKUs', fontsize=16)

for i, sku in enumerate(top_skus):
    sku_data = df[df['sku'] == sku].set_index('date')['demand'].sort_index()
    
    # Perform decomposition
    decomposition = seasonal_decompose(sku_data, model='additive', period=52)
    
    # Plot components
    decomposition.observed.plot(ax=axes[i, 0], title=f'{sku} - Observed')
    decomposition.trend.plot(ax=axes[i, 1], title=f'{sku} - Trend')
    decomposition.seasonal.plot(ax=axes[i, 2], title=f'{sku} - Seasonal')
    decomposition.resid.plot(ax=axes[i, 3], title=f'{sku} - Residual')

plt.tight_layout()
plt.show()

## 5. Autocorrelation Analysis

In [ ]:
# ACF and PACF plots for top SKUs
fig, axes = plt.subplots(len(top_skus), 2, figsize=(15, 12))
fig.suptitle('ACF and PACF Analysis - Top SKUs', fontsize=16)

for i, sku in enumerate(top_skus):
    sku_data = df[df['sku'] == sku].set_index('date')['demand'].sort_index()
    
    # ACF
    acf_vals = acf(sku_data, nlags=40, fft=True)
    axes[i, 0].plot(acf_vals)
    axes[i, 0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[i, 0].axhline(y=1.96/np.sqrt(len(sku_data)), color='r', linestyle='--', alpha=0.5)
    axes[i, 0].axhline(y=-1.96/np.sqrt(len(sku_data)), color='r', linestyle='--', alpha=0.5)
    axes[i, 0].set_title(f'{sku} - ACF')
    
    # PACF
    pacf_vals = pacf(sku_data, nlags=40)
    axes[i, 1].plot(pacf_vals)
    axes[i, 1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[i, 1].axhline(y=1.96/np.sqrt(len(sku_data)), color='r', linestyle='--', alpha=0.5)
    axes[i, 1].axhline(y=-1.96/np.sqrt(len(sku_data)), color='r', linestyle='--', alpha=0.5)
    axes[i, 1].set_title(f'{sku} - PACF')

plt.tight_layout()
plt.show()

## 6. External Factors Analysis

In [ ]:
# Holiday effect analysis
holiday_effect = df.groupby(['sku', 'is_holiday'])['demand'].mean().unstack()
holiday_effect['holiday_lift'] = (holiday_effect[True] / holiday_effect[False] - 1) * 100

plt.figure(figsize=(12, 6))
holiday_effect['holiday_lift'].plot(kind='bar')
plt.title('Holiday Demand Lift by SKU (%)')
plt.ylabel('Holiday Lift (%)')
plt.xticks(rotation=45)
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Promotion effect analysis
promo_effect = df.groupby(['sku', 'is_promotion'])['demand'].mean().unstack()
promo_effect['promo_lift'] = (promo_effect[True] / promo_effect[False] - 1) * 100

plt.figure(figsize=(12, 6))
promo_effect['promo_lift'].plot(kind='bar')
plt.title('Promotion Demand Lift by SKU (%)')
plt.ylabel('Promotion Lift (%)')
plt.xticks(rotation=45)
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Anomaly Detection

In [ ]:
# Identify anomalies using IQR method
def detect_anomalies_iqr(data, k=1.5):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - k * IQR
    upper_bound = Q3 + k * IQR
    return (data < lower_bound) | (data > upper_bound)

# Detect anomalies for each SKU
anomalies = df.groupby('sku')['demand'].apply(detect_anomalies_iqr).reset_index()
df['is_anomaly'] = anomalies['demand']

print(f"Total anomalies detected: {df['is_anomaly'].sum()}")
print(f"Percentage of anomalies: {df['is_anomaly'].mean()*100:.2f}%")

In [ ]:
# Visualize anomalies for top SKU
top_sku = top_skus[0]
sku_data = df[df['sku'] == top_sku]

fig = go.Figure()
fig.add_trace(go.Scatter(x=sku_data['date'], y=sku_data['demand'],
                        mode='lines+markers', name='Normal Data',
                        marker=dict(color='blue')))

anomalies_data = sku_data[sku_data['is_anomaly']]
if len(anomalies_data) > 0:
    fig.add_trace(go.Scatter(x=anomalies_data['date'], y=anomalies_data['demand'],
                            mode='markers', name='Anomalies',
                            marker=dict(color='red', size=10)))

fig.update_layout(title=f'Anomaly Detection - {top_sku}',
                 xaxis_title='Date', yaxis_title='Demand')
fig.show()

## 8. Statistical Tests

In [ ]:
# Ljung-Box test for autocorrelation in residuals
print("Ljung-Box Test Results (p-values):")
for sku in top_skus:
    sku_data = df[df['sku'] == sku].set_index('date')['demand'].sort_index()
    lb_stat, lb_pvalue = acorr_ljungbox(sku_data, lags=10, return_df=False)
    print(f"{sku}: {lb_pvalue[-1]:.4f}")

In [ ]:
# Normality test for demand distribution
print("Shapiro-Wilk Normality Test Results:")
for sku in top_skus[:4]:  # Test top 4 SKUs
    sku_data = df[df['sku'] == sku]['demand']
    stat, p_value = stats.shapiro(sku_data)
    print(f"{sku}: statistic={stat:.4f}, p-value={p_value:.4f}")

## 9. Summary and Insights

In [ ]:
# Generate summary statistics
summary_stats = df.groupby('sku')['demand'].agg([
    'count', 'mean', 'std', 'min', 'max', 
    lambda x: x.quantile(0.25),
    lambda x: x.quantile(0.75)
]).round(2)

summary_stats.columns = ['Count', 'Mean', 'Std', 'Min', 'Max', 'Q25', 'Q75']
summary_stats['CV'] = (summary_stats['Std'] / summary_stats['Mean']).round(3)

print("Summary Statistics by SKU:")
print(summary_stats)

## Key Findings:

1. **Seasonal Patterns**: Strong seasonal variations observed across SKUs
2. **Trend Analysis**: Long-term growth trends identified
3. **External Factors**: Holiday and promotion effects vary by SKU
4. **Autocorrelation**: Significant lag dependencies detected
5. **Data Quality**: Anomalies identified and flagged

These insights will inform the forecasting model selection and feature engineering process.